# 人流: 直近 1 時間の徒歩ユニーク人数を 25 m メッシュで数える

`probe_trips.py` が書いた `*_points.csv`（全点 + Stay/Move・モードの判定列。利用者 ID を含む唯一の出力）を読み、
15 分スロットごとに「直近 1 時間（(T−60 分, T]）にそのセルに徒歩の点を 1 つでも持つ利用者 ID の数」を 25 m メッシュで数えます。
平時・有事それぞれの件数と、その比（有事 / 平時）を GeoTIFF に書き、ビューワーのグリッドタブ（入力 6）でそのまま読めます。
このノートブックは probe_trips.py を import しません（必要なのは numpy と pandas だけ）。

| 節 | 処理 | 出力 |
|---|---|---|
| 1 | 入力ファイルの役割（平時／有事）とスロット軸を決める | – |
| 2 | 点を少しずつ読み、徒歩の点だけを残す（線分を結ぶための列も持つ） | – |
| 3 | メッシュを決め、線分を刻んで、スロット × セルの利用者 ID 数を数える | – |
| 4 | GeoTIFF と index.json を書く | `walk25_{baseline,event}_HHMM.tif`, `walk25_HHMM.tif`, `index.json` |
| 5 | 比が平時の 1/3 以下または 3 倍以上のセルを、窓ごとに GeoJSON（セルの四角形）に書く | `anomaly/walk25_anomaly_HHMM.geojson` |

数え方

- 徒歩の点 = `segment == "Move"` かつ `dense == 1` かつ `mode == "walk"`（ビューワーの軌跡タブに描かれる点と同じ）。`WALK_ONLY = False` で全点
- 窓は (T − `WINDOW_MIN`, T]、T は `SLOT_MIN` 刻み。同じ利用者がその窓・そのセルに何点あっても 1 と数える
- 点だけでなく、軌跡が通過したセルにも利用者を割り付ける（`TRACE_SEGMENTS = True`）。同じ利用者の連続する徒歩の点（同じトリップ、間に別の点が無く、間隔 `LINK_MAX_GAP_MIN` 以内、歩行ジャンプで切れていない = ビューワーが線で結ぶ対）を結ぶ線分を `SAMPLE_M` ごとに刻み、刻んだ点の時刻は両端から線形補間する
- 平時が複数日あれば日平均。比は平時の人数が `MIN_BASE_USERS`（既定 5 人）以上のセルだけ出し、未満は NaN
- 入力の役割: ファイル名が `baseline_…` / `event_…`（probe_trips の期間モードの出力）ならその通り、`YYYYMMDD_points.csv`（日別モード）なら `EVENT_DATE` の日が有事、他が平時
- スロットの並び: 期間モードの出力は期間の開始時刻から 15 分刻み（12:00, 12:15, …, 11:45）、日別モードは 00:15 … 24:00（窓の終端の時刻）
- メッシュの範囲: `GRID_BBOX`、無ければ `GRID_INDEX`（probe_trips の `grid/index.json` の bounds と同じ範囲）、どちらも無ければ徒歩の点の範囲

25 m セルは 100 m の 16 倍の数になります。範囲 10 km × 10 km で 16 万セル（1 枚 0.6 MB）、96 スロット × 3 枚で約 200 MB。
東京都全域のような広い範囲は `GRID_BBOX` で対象を絞ってください。

## 0. パラメータ

In [ ]:
from pathlib import Path
import json, math, re, struct, time
import numpy as np
import pandas as pd

# ---- 入出力 -------------------------------------------------------------
POINTS_FILES = sorted(Path("./probe_out/2024").glob("*_points.csv"))   # probe_trips.py の *_points.csv（平時・有事の両方）
EVENT_DATE   = "2024-08-21"                 # 日別モードの出力（YYYYMMDD_points.csv）のとき、この日が有事、他が平時
OUT_DIR      = Path("./probe_out/2024/grid_walk25")   # 出力先（ビューワーの入力 6、または人流フォルダの中の grid*/ として読まれる）
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- メッシュと窓 ---------------------------------------------------------
CELL_M     = 25.0                           # セルの一辺 [m]
GRID_BBOX  = None                           # (lon_min, lat_min, lon_max, lat_max)。None なら GRID_INDEX、それも無ければ点の範囲
GRID_INDEX = Path("./probe_out/2024/grid/index.json")   # 範囲を揃えたい既存の index.json（無ければ無視）
SLOT_MIN   = 15                             # スロット [分]
WINDOW_MIN = 60                             # 窓 [分]（直近 1 時間）
MIN_BASE_USERS = 5.0                        # 比（有事 / 平時）は平時の人数（日平均）がこれ以上のセルだけ。未満は NaN
RATIO_LOW  = 1 / 3                          # 比がこれ以下のセルを「減少」として GeoJSON に出す
RATIO_HIGH = 3.0                            # 比がこれ以上のセルを「増加」として GeoJSON に出す
PERIOD_HOURS = 24                           # 期間モードの出力のとき、期間の長さ [時間]（probe_trips の PERIOD_HOURS）

# ---- 点の選び方と線分 ----------------------------------------------------
WALK_ONLY  = True                           # True: Move かつ dense かつ walk の点だけ。False: 全点
TRACE_SEGMENTS = True                       # True: 連続する点を結ぶ線分が通過したセルにも利用者を割り付ける。False: 点のセルだけ
SAMPLE_M   = 5.0                            # 線分を刻む間隔 [m]（セルの一辺の 1/5 以下にする）
LINK_MAX_GAP_MIN = 2.0                      # 連続する点をつなぐ最大時間差 [分]（probe_trips の DENSE_MAX_GAP_MIN と同じ値）
CHUNK_ROWS = 1_000_000                      # CSV を一度に読む行数

PARAM, LABEL, UNIT = "walk25", "徒歩ユニーク人数（25 m）", "人"   # ビューワーの層の名前・表示
NODATA = -99.0                              # 件数ラスタの nodata（比は NaN）

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

## 1. 入力ファイルの役割とスロット軸

ファイル名から役割（baseline / event）と、期間モードなら期間の開始時刻を取ります。
スロット k の終端 T_k = day0 + k × SLOT_MIN。day0 は期間の開始時刻（期間モード）か、その日の 0 時（日別モード）です。

In [ ]:
def describe(path):
    """(role, day0, k_lo, k_hi) をファイル名から決める。"""
    m = re.match(r"(baseline|event)_(\d{8})_(\d{4})_points\.csv$", path.name)
    if m:                                                          # 期間モード: 期間の開始から 15 分刻み
        day0 = pd.Timestamp(f"{m.group(2)} {m.group(3)[:2]}:{m.group(3)[2:]}")
        return m.group(1), day0, 0, int(round(PERIOD_HOURS * 60 / SLOT_MIN)) - 1
    m = re.match(r"(\d{8})_points\.csv$", path.name)
    if not m:
        raise ValueError(f"ファイル名から日付を読めません: {path.name}")
    day0 = pd.Timestamp(m.group(1))                                # 日別モード: 00:15 … 24:00
    role = "event" if day0.date() == pd.Timestamp(EVENT_DATE).date() else "baseline"
    return role, day0, 1, int(1440 / SLOT_MIN)

jobs = [(p, *describe(p)) for p in POINTS_FILES]
if not jobs:
    raise FileNotFoundError("POINTS_FILES が空です")
for p, role, day0, k_lo, k_hi in jobs:
    print(f"{p.name}: {role}, day0 {day0}, slots {k_lo}..{k_hi}")
label_of = lambda day0, k: (day0 + pd.Timedelta(minutes=k * SLOT_MIN)).strftime("%H:%M")
N_WIN = int(round(WINDOW_MIN / SLOT_MIN))                          # 1 点が属する窓の数

## 2. 点を読む

`CHUNK_ROWS` 行ずつ読み、徒歩の点だけを残して (利用者コード, 時刻, lon, lat, トリップ番号, 歩行ジャンプ, 行番号) にします。
利用者 ID はファイル内で整数コードに置き換え、出力には出しません。行番号は「間に別の点が無い」ことを見るために持ちます。

In [ ]:
def read_walk_points(path):
    """徒歩の点を dict of numpy 配列で返す: ucode（ファイル内の通し番号）, t, lon, lat, trip, wbreak, row（元の行番号）。"""
    cols = ["userid", "recordedat", "lon", "lat", "trip_no", "walk_break"] + (["segment", "dense", "mode"] if WALK_ONLY else [])
    uids, parts, offset = {}, [], 0
    for ch in pd.read_csv(path, usecols=cols, chunksize=CHUNK_ROWS, dtype={"userid": str}):
        row = np.arange(offset, offset + len(ch)); offset += len(ch)
        if WALK_ONLY:
            keep = ((ch["segment"] == "Move") & (ch["dense"] == 1) & (ch["mode"] == "walk")).to_numpy()
            ch, row = ch[keep], row[keep]
        if ch.empty:
            continue
        codes = np.fromiter((uids.setdefault(u, len(uids)) for u in ch["userid"]), dtype=np.int64, count=len(ch))
        parts.append((codes, pd.to_datetime(ch["recordedat"]).to_numpy(), ch["lon"].to_numpy(float), ch["lat"].to_numpy(float),
                      ch["trip_no"].to_numpy(np.int64), ch["walk_break"].to_numpy(np.int8), row))
    names = ["ucode", "t", "lon", "lat", "trip", "wbreak", "row"]
    if not parts:
        return dict(zip(names, [np.zeros(0, np.int64), np.zeros(0, "datetime64[ns]"), np.zeros(0), np.zeros(0), np.zeros(0, np.int64), np.zeros(0, np.int8), np.zeros(0, np.int64)]))
    P = dict(zip(names, (np.concatenate(x) for x in zip(*parts))))
    log(f"  {path.name}: 徒歩の点 {len(P['ucode']):,}、利用者 {len(uids):,}")
    return P

points = {}
for p, role, day0, k_lo, k_hi in jobs:
    t0 = time.perf_counter()
    points[p] = read_walk_points(p)
    log(f"  read in {time.perf_counter() - t0:.1f} s")

## 3. メッシュとスロット × セルの利用者数

セルは経緯度の格子（中心緯度で一辺 `CELL_M` m）。行 0 が北端、セル番号 = 行 × 列数 + 列（GeoTIFF の並び）。
`TRACE_SEGMENTS` なら、まず連続する点を結ぶ線分を `SAMPLE_M` ごとに刻んで点を増やします（時刻は線形補間）。
次に各点を、それを含む `N_WIN` 個の窓（終端 T_k が点の時刻以上、T_k − WINDOW_MIN が点の時刻未満）に複製し、(窓, セル, 利用者) の重複を消してから数えます。
メモリを抑えるため、利用者のブロックごとに処理して合計します（利用者はブロックをまたがないので、ユニーク数をそのまま足せます）。

In [ ]:
M_PER_DEG = 6371008.8 * math.pi / 180.0

class Mesh:
    def __init__(self, bbox, cell_m):
        west, south, east, north = (float(v) for v in bbox)
        self.dy = cell_m / M_PER_DEG
        self.dx = cell_m / (M_PER_DEG * math.cos(math.radians((south + north) / 2)))
        self.ncol = max(1, int(math.ceil((east - west) / self.dx)))
        self.nrow = max(1, int(math.ceil((north - south) / self.dy)))
        self.west, self.north, self.cell_m = west, north, cell_m
        self.east, self.south = west + self.ncol * self.dx, north - self.nrow * self.dy
    def cell(self, lon, lat):
        col = np.floor((lon - self.west) / self.dx); row = np.floor((self.north - lat) / self.dy)
        ok = (col >= 0) & (col < self.ncol) & (row >= 0) & (row < self.nrow)
        return np.where(ok, row * self.ncol + col, -1).astype(np.int64)

def mesh_bbox():
    if GRID_BBOX:
        return tuple(GRID_BBOX), "GRID_BBOX"
    if GRID_INDEX and Path(GRID_INDEX).exists():
        return tuple(json.load(open(GRID_INDEX, encoding="utf-8"))["bounds"]), str(GRID_INDEX)
    lons = np.concatenate([v["lon"] for v in points.values()]); lats = np.concatenate([v["lat"] for v in points.values()])
    if not len(lons):
        raise ValueError("徒歩の点がありません")
    return (lons.min(), lats.min(), lons.max(), lats.max()), "点の範囲"

bbox, src = mesh_bbox()
mesh = Mesh(bbox, CELL_M)
log(f"メッシュ: {mesh.ncol} x {mesh.nrow} セル（{CELL_M:g} m、範囲は {src}）")

def trace_segments(P, sl):
    """ブロック sl の点に、連続する点を結ぶ線分を SAMPLE_M ごとに刻んだ点を加えて (ucode, mins, lon, lat) を返す（mins は day0 からの分）。"""
    u, mins, lon, lat = P["ucode"][sl], P["mins"][sl], P["lon"][sl], P["lat"][sl]
    if not TRACE_SEGMENTS or len(u) < 2:
        return u, mins, lon, lat
    trip, wb, row = P["trip"][sl], P["wbreak"][sl], P["row"][sl]
    link = ((u[1:] == u[:-1]) & (trip[1:] == trip[:-1]) & (row[1:] == row[:-1] + 1) & (wb[1:] == 0)   # ビューワーが線で結ぶ対
            & ((mins[1:] - mins[:-1]) <= LINK_MAX_GAP_MIN) & ((mins[1:] - mins[:-1]) >= 0))
    i = np.flatnonzero(link)                                                   # 線分 i: 点 i -> 点 i+1
    kx = M_PER_DEG * math.cos(math.radians(mesh.north - (mesh.north - mesh.south) / 2))
    seg_m = np.hypot((lon[i + 1] - lon[i]) * kx, (lat[i + 1] - lat[i]) * M_PER_DEG)
    n_in = np.floor(seg_m / SAMPLE_M).astype(np.int64)                          # 内部の刻み点の数（両端は元の点）
    n_in = np.where(n_in * SAMPLE_M >= seg_m, n_in - 1, n_in).clip(min=0)
    tot = int(n_in.sum())
    if tot == 0:
        return u, mins, lon, lat
    seg = np.repeat(i, n_in)
    f = (np.arange(tot) - np.repeat(np.cumsum(n_in) - n_in, n_in) + 1) / np.repeat(n_in + 1, n_in)   # 0 < f < 1
    return (np.concatenate([u, u[seg]]), np.concatenate([mins, mins[seg] + f * (mins[seg + 1] - mins[seg])]),
            np.concatenate([lon, lon[seg] + f * (lon[seg + 1] - lon[seg])]), np.concatenate([lat, lat[seg] + f * (lat[seg + 1] - lat[seg])]))

def count_users(P, day0, k_lo, k_hi, block_points=2_000_000):
    """{k: Series(cell -> 利用者数)}: 窓 (T_k - WINDOW_MIN, T_k] にセル内の点（刻んだ点を含む）を持つ利用者の数。"""
    P["mins"] = (P["t"] - day0.to_datetime64()) / np.timedelta64(1, "m")
    n = len(P["ucode"]); out = {}; n_samples = 0
    starts = [0]
    while starts[-1] < n:                                                      # 利用者の境界でブロックを切る
        e = min(starts[-1] + block_points, n)
        while e < n and P["ucode"][e] == P["ucode"][e - 1]:
            e += 1
        starts.append(e)
    for a, b in zip(starts[:-1], starts[1:]):
        u, mins, lon, lat = trace_segments(P, slice(a, b)); n_samples += len(u) - (b - a)
        k0 = np.maximum(np.ceil(mins / SLOT_MIN).astype(np.int64), k_lo)       # 点の時刻以上の最初の窓終端
        cells = mesh.cell(lon, lat)
        key = pd.DataFrame({"k0": k0, "cell": cells, "u": u})[cells >= 0].drop_duplicates()   # 同じ窓の組になる点をここで減らす
        rep = np.repeat(np.arange(len(key)), N_WIN)
        kk = np.repeat(key["k0"].to_numpy(), N_WIN) + np.tile(np.arange(N_WIN), len(key))
        ok = (kk <= k_hi) & (kk * SLOT_MIN - WINDOW_MIN < np.repeat(mins[key.index.to_numpy()], N_WIN))   # 窓が点を含む
        kc = pd.DataFrame({"k": kk[ok], "cell": key["cell"].to_numpy()[rep[ok]], "u": key["u"].to_numpy()[rep[ok]]}).drop_duplicates()
        for k, g in kc.groupby("k"):
            c = g.groupby("cell").size().astype(np.float32)
            out[int(k)] = c if int(k) not in out else out[int(k)].add(c, fill_value=0.0)
    if TRACE_SEGMENTS:
        log(f"  線分を刻んだ点 {n_samples:,}（元の点 {n:,}）")
    return out

acc, days, labels = {}, {}, []                                     # (role, label) -> Series(cell -> 日合計), role -> 日数, ラベルの並び
for p, role, day0, k_lo, k_hi in jobs:
    t0 = time.perf_counter()
    counts = count_users(points[p], day0, k_lo, k_hi)
    for k in range(k_lo, k_hi + 1):
        lab = label_of(day0, k)
        if lab not in labels:
            labels.append(lab)
        if k in counts:
            acc[(role, lab)] = counts[k] if (role, lab) not in acc else acc[(role, lab)].add(counts[k], fill_value=0.0)
    days[role] = days.get(role, 0) + 1
    log(f"  {role} {p.name}: {sum(len(v) for v in counts.values()):,} (窓, セル) 組 in {time.perf_counter() - t0:.1f} s")
print("役割と日数:", days, "/ スロット", len(labels), "個:", labels[:4], "…", labels[-2:])

## 4. GeoTIFF と index.json

`walk25_baseline_HHMM.tif` / `walk25_event_HHMM.tif`（float32 の人数、nodata −99）と、両方あるとき `walk25_HHMM.tif`（有事 / 平時。平時が `MIN_BASE_USERS` 未満のセルは NaN）。
`index.json` はビューワーが層の名前・単位・窓の長さを読むためのものです（probe_trips の `grid/index.json` と同じ形式）。

In [ ]:
def write_geotiff(path, arr, west, north, dx, dy, nodata):
    """非圧縮 float32 GeoTIFF、EPSG:4326、行 0 が北（probe_trips.py と同じ書き方）。"""
    h, w = arr.shape
    data = arr.astype("<f4").tobytes()
    ascii_nodata = (str(int(nodata)) if float(nodata).is_integer() else str(nodata)).encode() + b"\x00"
    geokeys = struct.pack("<16H", 1, 1, 0, 3, 1024, 0, 1, 2, 1025, 0, 1, 1, 2048, 0, 1, 4326)
    pixel_scale = struct.pack("<3d", dx, dy, 0.0)
    tiepoint = struct.pack("<6d", 0.0, 0.0, 0.0, west, north, 0.0)
    tags = [(256, 4, 1, w), (257, 4, 1, h), (258, 3, 1, 32), (259, 3, 1, 1), (262, 3, 1, 1), (273, 4, 1, None),
            (277, 3, 1, 1), (278, 4, 1, h), (279, 4, 1, len(data)), (284, 3, 1, 1), (339, 3, 1, 3),
            (33550, 12, 3, pixel_scale), (33922, 12, 6, tiepoint), (34735, 3, 16, geokeys), (42113, 2, len(ascii_nodata), ascii_nodata)]
    tags.sort()
    extra_off = 8 + 2 + 12 * len(tags) + 4
    data_off = extra_off + sum(len(v) for _, _, _, v in tags if isinstance(v, bytes) and len(v) > 4)
    extra, entries = b"", b""
    for tag, typ, count, value in tags:
        if tag == 273:
            value = data_off
        if isinstance(value, bytes):
            if len(value) > 4:
                entries += struct.pack("<HHII", tag, typ, count, extra_off + len(extra)); extra += value
            else:
                entries += struct.pack("<HHI", tag, typ, count) + value.ljust(4, b"\x00")
        elif typ == 3:
            entries += struct.pack("<HHIHH", tag, typ, count, value, 0)
        else:
            entries += struct.pack("<HHII", tag, typ, count, value)
    with open(path, "wb") as f:
        f.write(b"II*\x00" + struct.pack("<I", 8))
        f.write(struct.pack("<H", len(tags)) + entries + struct.pack("<I", 0))
        f.write(extra)
        f.write(data)

def dense(role, lab):
    arr = np.zeros(mesh.nrow * mesh.ncol, dtype=np.float32)
    v = acc.get((role, lab))
    if v is not None:
        arr[v.index.to_numpy()] = v.to_numpy(np.float32) / days[role]     # 日平均
    return arr.reshape(mesh.nrow, mesh.ncol)

for old in OUT_DIR.glob(f"{PARAM}_*.tif"):
    old.unlink()
files = []
both = {"baseline", "event"} <= set(days)
for lab in labels:
    hhmm = lab.replace(":", "")
    b, e = dense("baseline", lab), dense("event", lab)
    for role, arr in (("baseline", b), ("event", e)):
        if role in days:
            name = f"{PARAM}_{role}_{hhmm}.tif"
            write_geotiff(OUT_DIR / name, arr, mesh.west, mesh.north, mesh.dx, mesh.dy, NODATA); files.append(name)
    if both:
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(b >= MIN_BASE_USERS, e / b, np.nan).astype(np.float32)
        name = f"{PARAM}_{hhmm}.tif"
        write_geotiff(OUT_DIR / name, ratio, mesh.west, mesh.north, mesh.dx, mesh.dy, float("nan")); files.append(name)

index = {"cell_m": CELL_M, "bounds": [mesh.west, mesh.south, mesh.east, mesh.north], "width": mesh.ncol, "height": mesh.nrow,
         "dx": mesh.dx, "dy": mesh.dy, "nodata": "nan", "count_nodata": NODATA, "params": [PARAM],
         "labels": {PARAM: LABEL}, "units": {PARAM: UNIT}, "window_min": WINDOW_MIN, "slot_min": SLOT_MIN,
         "slots": labels, "roles": sorted(days), "days": days, "walk_only": WALK_ONLY, "trace_segments": TRACE_SEGMENTS,
         "sample_m": SAMPLE_M, "min_base_users": MIN_BASE_USERS,
         "values": f"{PARAM}_<role>_<HHMM>: 窓 (T - {WINDOW_MIN} 分, T] にセル内の徒歩の点{'（軌跡が通過したセルを含む）' if TRACE_SEGMENTS else ''}を持つ利用者 ID の数（日平均）; "
                   f"{PARAM}_<HHMM>: 有事 / 平時（平時 {MIN_BASE_USERS:g} 人未満は NaN）", "files": files}
with open(OUT_DIR / "index.json", "w", encoding="utf-8") as f:
    json.dump(index, f, ensure_ascii=False, indent=1)
log(f"wrote {len(files)} rasters to {OUT_DIR}（{len(labels)} スロット、{mesh.ncol * mesh.nrow:,} セル、役割 {sorted(days)}）")

## 5. 比が大きく変わったセルの GeoJSON（窓ごと）

比（有事 / 平時。平時 `MIN_BASE_USERS` 人以上のセルだけ）が `RATIO_LOW`（1/3）以下か `RATIO_HIGH`（3 倍）以上のセルを、
窓ごとに `anomaly/walk25_anomaly_HHMM.geojson` に書きます。地物はセルの四角形（Polygon）で、属性は
cell（セル番号）, row, col, time（窓の終端 "HH:MM"）, baseline, event（人数、日平均）, ratio, kind（"low" = 減少 / "high" = 増加）。
該当が無い窓も空の FeatureCollection を書くので、ファイル数は常にスロット数と同じです。平時・有事の両方が無いときは書きません。

In [ ]:
ANOM_DIR = OUT_DIR / "anomaly"; ANOM_DIR.mkdir(exist_ok=True)
for old in ANOM_DIR.glob(f"{PARAM}_anomaly_*.geojson"):
    old.unlink()

def cell_polygon(cells):
    """セル番号 -> [[lon, lat] x 5]（左下から反時計回り）"""
    row, col = cells // mesh.ncol, cells % mesh.ncol
    x0 = mesh.west + col * mesh.dx; x1 = x0 + mesh.dx
    y1 = mesh.north - row * mesh.dy; y0 = y1 - mesh.dy
    return [[[round(float(a), 7), round(float(b), 7)] for a, b in ((x0[i], y0[i]), (x1[i], y0[i]), (x1[i], y1[i]), (x0[i], y1[i]), (x0[i], y0[i]))]
            for i in range(len(cells))]

anom_counts = []
if both:
    for lab in labels:
        hhmm = lab.replace(":", "")
        b, e = dense("baseline", lab).ravel(), dense("event", lab).ravel()
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(b >= MIN_BASE_USERS, e / b, np.nan)
        hit = np.flatnonzero((ratio <= RATIO_LOW) | (ratio >= RATIO_HIGH))
        polys = cell_polygon(hit)
        feats = [{"type": "Feature",
                  "properties": {"cell": int(c), "row": int(c // mesh.ncol), "col": int(c % mesh.ncol), "time": lab,
                                 "baseline": round(float(b[c]), 3), "event": round(float(e[c]), 3), "ratio": round(float(ratio[c]), 4),
                                 "kind": "low" if ratio[c] <= RATIO_LOW else "high"},
                  "geometry": {"type": "Polygon", "coordinates": [polys[i]]}}
                 for i, c in enumerate(hit)]
        with open(ANOM_DIR / f"{PARAM}_anomaly_{hhmm}.geojson", "w", encoding="utf-8") as f:
            json.dump({"type": "FeatureCollection", "name": f"{PARAM}_anomaly_{hhmm}",
                       "properties": {"time": lab, "window_min": WINDOW_MIN, "cell_m": CELL_M, "min_base_users": MIN_BASE_USERS,
                                      "ratio_low": RATIO_LOW, "ratio_high": RATIO_HIGH},
                       "features": feats}, f, ensure_ascii=False)
        anom_counts.append({"time": lab, "low": int((ratio[hit] <= RATIO_LOW).sum()), "high": int((ratio[hit] >= RATIO_HIGH).sum())})
    anom_counts = pd.DataFrame(anom_counts).set_index("time")
    log(f"wrote {len(anom_counts)} files to {ANOM_DIR}（減少 {int(anom_counts.low.sum()):,} セル、増加 {int(anom_counts.high.sum()):,} セル）")
    display(anom_counts[(anom_counts.low > 0) | (anom_counts.high > 0)])
else:
    print("平時・有事の両方が無いので比の GeoJSON は書きません")